# 📡 01 - Multi-Modal Ingestion: RSS Feeds & YouTube Audio

### Pipeline Stage 1: Data Ingestion & Normalization
This notebook demonstrates our dual-modality ingestion engine:
1. **Live and pre-cached RSS feeds** spanning 10 institutional, retail, and tech media outlets.
2. **YouTube audio extraction and local Whisper transcription** for financial influencer video content.

---

In [1]:
# Ensure project root is in sys.path and is current working directory
import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print(f'[OK] Working directory set to project root: {PROJECT_ROOT}')

import pandas as pd
from src.ingest import RSSIngester, YouTubeIngester, ingest_all

# Ingest sample multimodal dataset (offline reproducible mode)
df_raw = ingest_all(use_sample=True)
print(f'Total items ingested: {len(df_raw)}')
print(f'Modality breakdown:\n{df_raw["source_type"].value_counts()}')
df_raw.head(3)

[OK] Working directory set to project root: C:\Users\RANADEEP\Documents\BS & Hype Analyzer project


Total items ingested: 151
Modality breakdown:
source_type
rss                 150
audio_transcript      1
Name: count, dtype: int64


,id,outlet,outlet_id,category,bias_label,source_type,title,summary,link,published,author,outlet_weight,baseline_expected_hype,full_text
0,bloomberg_tech_0001,Bloomberg Technology,bloomberg_tech,Technology & Finance,Institutional Financial,rss,Apple Expands AI Infrastructure With Multi-Bil...,Apple Inc. is committing $5 billion over three...,https://example.com/bloomberg_tech/bloomberg_t...,2026-09-10T12:00:00Z,Bloomberg Technology Staff,1.0,0.18,NaN
1,bloomberg_tech_0002,Bloomberg Technology,bloomberg_tech,Technology & Finance,Institutional Financial,rss,Nvidia Revenue Growth Moderates to 45% as Clou...,Nvidia Corp. reported quarterly sales of $32.4...,https://example.com/bloomberg_tech/bloomberg_t...,2026-09-10T12:00:00Z,Bloomberg Technology Staff,1.0,0.22,NaN
2,bloomberg_tech_0003,Bloomberg Technology,bloomberg_tech,Technology & Finance,Institutional Financial,rss,TSMC Accelerates 2nm Production Timeline Ahead...,Taiwan Semiconductor Manufacturing Co. confirm...,https://example.com/bloomberg_tech/bloomberg_t...,2026-09-10T12:00:00Z,Bloomberg Technology Staff,1.0,0.15,NaN


## 2. Outlet Representation & Volume Analysis

Let's inspect how articles are distributed across media categories and bias labels.

In [2]:
outlet_summary = df_raw.groupby(['outlet', 'category', 'bias_label']).size().reset_index(name='article_count')
outlet_summary.sort_values(by='article_count', ascending=False)

,outlet,category,bias_label,article_count
0,Bloomberg Technology,Technology & Finance,Institutional Financial,15
1,CNBC Markets,Financial Media,Sensational Financial Broadcast,15
2,CoinDesk Crypto News,Crypto & Web3,Speculative Crypto,15
4,Financial Times Markets,Institutional Finance,Institutional Analysis,15
5,MarketWatch Bulletins,Retail Trading,Retail Momentum,15
6,Reuters Business,Macro & Markets,Wire Service (Baseline),15
7,TechCrunch AI & Startups,Tech Startups,Venture Hype Cycle,15
9,Wall Street Journal Markets,Financial News,Mainstream Financial,15
8,The Verge Tech,Tech Journalism,Consumer Tech Critique,15
10,Yahoo Finance,Retail Finance,Retail Aggregator,15


## 3. Multi-Modal Audio Ingestion: Local Whisper Transcription

Financial hype frequently spreads via video and podcast channels before hitting print. Our audio pipeline ingests YouTube audio tracks via `yt-dlp` and generates transcriptions locally using OpenAI Whisper without sending any data to third-party cloud APIs.

In [3]:
yt_sample = YouTubeIngester.load_cached_transcript()
print(f'Video Title:        {yt_sample["title"]}')
print(f'Channel:            {yt_sample["channel"]}')
print(f'Speaker:            {yt_sample["speaker"]}')
print(f'Local Whisper Model:{yt_sample["whisper_model_used"]}')
print(f'Duration:           {yt_sample["duration_sec"]} seconds')
print('\n--- Transcript Excerpt ---')
print(yt_sample['full_transcript'][:400] + '...')
print('\n--- First 3 Timestamped Audio Segments ---')
for seg in yt_sample['segments'][:3]:
    print(f'[{seg["start"]:04.1f}s - {seg["end"]:04.1f}s] {seg["text"]}')

Video Title:        🚨 URGENT WARNING: The Secret AI Stock & Crypto Moonshot That Will 100x Overnight! (Bloodbath Incoming)
Channel:            Crypto Alpha Moonshots
Speaker:            Alpha Moonshots Host
Local Whisper Model:whisper-small (local)
Duration:           765 seconds

--- Transcript Excerpt ---
WHAT IS UP EVERYONE! Welcome back to Crypto Alpha Moonshots! URGENT WARNING: If you are holding any cash, stocks, or crypto right now, you MUST watch this video until the very end before it's too late! Insiders have just revealed an unprecedented paradigm shift in artificial intelligence and crypto that is guaranteed to send shockwaves across Wall Street! Top experts predict Bitcoin will 100x to $...

--- First 3 Timestamped Audio Segments ---
[00.0s - 06.5s] WHAT IS UP EVERYONE! Welcome back to Crypto Alpha Moonshots!
[06.5s - 15.0s] URGENT WARNING: If you are holding any cash, stocks, or crypto right now, you MUST watch this video until the very end before it's too late!
[15.0s - 

## 4. Persist Ingested Articles to Interim Cache

Save raw ingested articles to `data/interim/ingested_articles.csv` for downstream feature extraction.

In [4]:
from src.config import INTERIM_DATA_DIR

interim_file = INTERIM_DATA_DIR / 'ingested_articles.csv'
df_raw.to_csv(interim_file, index=False)
print(f'[OK] Successfully saved {len(df_raw)} records to {interim_file}')
print('Proceed to Notebook 02 for feature engineering and hype scoring!')

[OK] Successfully saved 151 records to C:\Users\RANADEEP\Documents\BS & Hype Analyzer project\data\interim\ingested_articles.csv
Proceed to Notebook 02 for feature engineering and hype scoring!
